Importing Basic Libraries

In [ ]:
import pandas as pd
import numpy as np

Importing Regression Funcitons

In [ ]:
from sklearn.linear_model import LinearRegression

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
import lightgbm as lgb

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

Importing Evaluator Functions

In [ ]:
!pip install sktime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 12.4 MB/s eta 0:00:00


In [ ]:
from sklearn.metrics import r2_score

from sklearn.metrics import mean_absolute_percentage_error
from sktime.performance_metrics.forecasting import median_absolute_percentage_error

Importing Classes and Functions from Alternate Files

In [ ]:
from model_classes import W4_Regression, W5_Regs, W7_Boosting

In [ ]:
# Preset file

filename = "CRMLS_0625-0626_enriched.csv"
end_mnth = 6

In [ ]:
def load_df(file=filename):
  """
  Takes a .csv filename or filepath
  Returns the DataFrame loaded from the csv
  """
  df = pd.read_csv(file, low_memory=False)
  df["CloseDate"] = pd.to_datetime(df["CloseDate"])     # Converts "CloseDate" values to datetime type
  return df

In [ ]:
enr_df = load_df()

In [ ]:
# Preset Values

main_cols = ["BedroomsTotal", "BathroomsTotalInteger", "LivingArea", "LotSizeSquareFeet",
             "DaysOnMarket", "YearBuilt", "PostalCode", "SaleMonth"]

extras = ["ViewYN", "FireplaceYN", "NewConstructionYN", "PoolPrivateYN"]
extra_cols = [a+"_True" for a in extras] + [a+"_False" for a in extras]

totals = main_cols+extra_cols


cols = enr_df.columns.to_list()
new_col_strt = [i for i in range(len(cols)) if cols[i] == "SaleMonth"]
new_cols = cols[(new_col_strt[0]+1):len(cols)]
new_cols = [i for i in new_cols if i != "DistrictName"]

crit_cols = totals+new_cols
log_cols = main_cols+new_cols


target = "ClosePrice"

In [ ]:
def save_csv(df, file):
  """
  Takes a DataFrame
  Saves the inputted DataFrame as a .csv file, given inputted name
  """
  df.to_csv(file, index=False)

In [ ]:
class W8_Eval():

  def __init__(self,df):
    self.df = df


  def mape_eval(self, y_test, y_pred):
    mape = mean_absolute_percentage_error(y_test, y_pred)       # Computes the mean absolute percentage error of y_test and the predicted y
    return mape


  def mdape_eval(self, y_test, y_pred):
    mdape = median_absolute_percentage_error(y_test, y_pred)       # Computes the median absolute percentage error of y_test and the predicted y
    return mdape

In [ ]:
base = W4_Regression(enr_df)
comp = W5_Regs(enr_df)
boost = W7_Boosting(enr_df)
Week8 = W8_Eval(enr_df)

train, test = base.test_train_split()


log_df = boost.log_transform()

bs = W4_Regression(log_df)
cp = W5_Regs(log_df)
bst = W7_Boosting(log_df)
Wk8 = W8_Eval(log_df)

tr, te = bs.test_train_split()

# **Evaluation:  *MAPE, MdAPE***

 * Mean Absolute Percentage Error (MAPE)
 * Median Absolute Percentage Error (MdAPE)

**Linear Regression**

In [ ]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_LRmape, LRmape = comp.shrt_main(base.LinReg, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logLRmape, logLRmape = cp.shrt_main(bs.LinReg, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True)

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_LRmdape, LRmdape = comp.shrt_main(base.LinReg, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logLRmdape, logLRmdape = cp.shrt_main(bs.LinReg, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True)

**Decision Tree Regressor**

In [ ]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_DTRmape, DTRmape = comp.shrt_main(comp.TreeReg, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logDTRmape, logDTRmape = cp.shrt_main(cp.TreeReg, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True)

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_DTRmdape, DTRmdape = comp.shrt_main(comp.TreeReg, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logDTRmdape, logDTRmdape = cp.shrt_main(cp.TreeReg, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True)

**Random Forest Regressor**

In [ ]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_RFRmape, RFRmape = comp.shrt_main(comp.ForestReg, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logRFRmape, logRFRmape = cp.shrt_main(cp.ForestReg, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True)

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_RFRmdape, RFRmdape = comp.shrt_main(comp.ForestReg, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True)

print("Log Transform")
c_logRFRmdape, logRFRmdape = cp.shrt_main(cp.ForestReg, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True)

In [ ]:
def load_data(file):
  """
  Takes a .csv filename or filepath
  Returns the DataFrame loaded from the csv
  """
  df = pd.read_csv(file, low_memory=False)
  return df

In [ ]:
bst_r2s = load_data("boosting_r2s.csv")

**XGBoost**

In [ ]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_XGBmape, XGBmape = comp.shrt_main(boost.XGB, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True,
                                    dep=bst_r2s["max_depth"].iloc[0], l_rate=bst_r2s["learning_rate"].iloc[0], est=bst_r2s["n_estimators"].iloc[0])

print("Log Transform")
c_logXGBmape, logXGBmape = cp.shrt_main(bst.XGB, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True,
                                        dep=bst_r2s["max_depth"].iloc[1], l_rate=bst_r2s["learning_rate"].iloc[1], est=bst_r2s["n_estimators"].iloc[1])

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_XGBmdape, XGBmdape = comp.shrt_main(boost.XGB, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True,
                                      dep=bst_r2s["max_depth"].iloc[0], l_rate=bst_r2s["learning_rate"].iloc[0], est=bst_r2s["n_estimators"].iloc[0])

print("Log Transform")
c_logXGBmdape, logXGBmdape = cp.shrt_main(bst.XGB, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True,
                                          dep=bst_r2s["max_depth"].iloc[1], l_rate=bst_r2s["learning_rate"].iloc[1], est=bst_r2s["n_estimators"].iloc[1])

**Gradient Boosting Regressor**

In [ ]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_GBRmape, GBRmape = comp.shrt_main(boost.GBR, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True,
                                    dep=bst_r2s["max_depth"].iloc[2], l_rate=bst_r2s["learning_rate"].iloc[2], est=bst_r2s["n_estimators"].iloc[2])

print("Log Transform")
c_logGBRmape, logGBRmape = cp.shrt_main(bst.GBR, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True,
                                        dep=bst_r2s["max_depth"].iloc[3], l_rate=bst_r2s["learning_rate"].iloc[3], est=bst_r2s["n_estimators"].iloc[3])

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
# print("Non-Transform")
c_GBRmdape, GBRmdape = comp.shrt_main(boost.GBR, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True,
                                      dep=bst_r2s["max_depth"].iloc[2], l_rate=bst_r2s["learning_rate"].iloc[2], est=bst_r2s["n_estimators"].iloc[2])

print("Log Transform")
c_logGBRmdape, logGBRmdape = cp.shrt_main(bst.GBR, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True,
                                          dep=bst_r2s["max_depth"].iloc[3], l_rate=bst_r2s["learning_rate"].iloc[3], est=bst_r2s["n_estimators"].iloc[3])

**LightGBM**

In [ ]:
# MAPE

In [ ]:
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_LGBMmape, LGBMmape = comp.shrt_main(boost.L_GBM, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True,
                                      dep=bst_r2s["max_depth"].iloc[4], l_rate=bst_r2s["learning_rate"].iloc[4], est=bst_r2s["n_estimators"].iloc[4])

In [ ]:
print("Log Transform")
c_logLGBMmape, logLGBMmape = cp.shrt_main(bst.L_GBM, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True,
                                          dep=bst_r2s["max_depth"].iloc[5], l_rate=bst_r2s["learning_rate"].iloc[5], est=bst_r2s["n_estimators"].iloc[5])

In [ ]:
# MdAPE

In [ ]:
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_LGBMmdape, LGBMmdape = comp.shrt_main(boost.L_GBM, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True,
                                        dep=bst_r2s["max_depth"].iloc[4], l_rate=bst_r2s["learning_rate"].iloc[4], est=bst_r2s["n_estimators"].iloc[4])

In [ ]:
print("Log Transform")
c_logLGBMmdape, logLGBMmdape = cp.shrt_main(bst.L_GBM, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True,
                                            dep=bst_r2s["max_depth"].iloc[5], l_rate=bst_r2s["learning_rate"].iloc[5], est=bst_r2s["n_estimators"].iloc[5])

Log Transform
[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=5) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=32) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=5) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=32) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002701 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14
[LightGBM] [Info] Number of data points in the train set: 92986, number of used features: 1
[LightGBM] [Info] Start training from score 13.785278
[LightGBM] [Warning] No further splits with 

**LightGBM Summary**

***Mean Absolute Percentage Error***

*Non-Transform*

* ***0.0305***

*Log Transform*

* ***0.0008***

***Median Absolute Percentage Error***

*Non-Transform*

* ***0.0207***

*Log Transform*

* ***0.0004***

**Histogram-based Gradient Boosting Regressor**

In [ ]:
# MAPE
print("Mean Absolute Percentage Error")
print("Non-Transform")
c_HGBRmape, HGBRmape = comp.shrt_main(boost.HGBR, train, test, Week8.mape_eval, crit_cols, rev=False, prt=True,
                                      dep=bst_r2s["max_depth"].iloc[6], l_rate=bst_r2s["learning_rate"].iloc[6])

print("Log Transform")
c_logHGBRmape, logHGBRmape = cp.shrt_main(bst.HGBR, tr, te, Wk8.mape_eval, crit_cols, rev=False, prt=True,
                                          dep=bst_r2s["max_depth"].iloc[7], l_rate=bst_r2s["learning_rate"].iloc[7])

print("\n")

# MdAPE
print("Median Absolute Percentage Error ")
print("Non-Transform")
c_HGBRmdape, HGBRmdape = comp.shrt_main(boost.HGBR, train, test, Week8.mdape_eval, crit_cols, rev=False, prt=True,
                                        dep=bst_r2s["max_depth"].iloc[6], l_rate=bst_r2s["learning_rate"].iloc[6])

print("Log Transform")
c_logHGBRmdape, logHGBRmdape = cp.shrt_main(bst.HGBR, tr, te, Wk8.mdape_eval, crit_cols, rev=False, prt=True,
                                            dep=bst_r2s["max_depth"].iloc[7], l_rate=bst_r2s["learning_rate"].iloc[7])

***Summary***

---

 Lowest MAPE:
   * Non-Transform: **.0013**
     * **Random Forest**
   * Log Transform: **0.0**
     * **Linear Regression**

---

 Lowest MdAPE:
   * Non-Transform: **0.0**
     * **Desicion Tree**
   * Log Transform: **0.0**
     * **Linear Regression / Decision Tree**

# **Price Bands**

In [ ]:
# Defines column "PriceRank" with .qcut from 'Low' to 'High'
enr_df["PriceRank"] = pd.qcut(enr_df["ClosePrice"], q=5, labels=["Low", "Low-Medium", "Medium", "Medium-High", "High"])

**Bin Values**
 * ***Q1: High***
   * `$1,610,000 - $110,000,000`
   * (1610000.0, 110000000.0]
 * ***Q2: Medium-High***
   * `$1,086,000 - $1,610,000`
   * (1086000.0, 1610000.0]
 * ***Q3: Medium***
   * `$791,000 - $1,086,000`
   * (791000.0, 1086000.0]
 * ***Q4: Low-Medium***
   * `$574,000 - $791,000`
   * (574000.0, 791000.0]
 * ***Q5: Low***
   * `$26,000 - $574,000`
   * (25999.999, 574000.0]

In [ ]:
# Bin Values
(sorted(pd.unique(pd.qcut(enr_df["ClosePrice"], q=5)), reverse=True))

[Interval(1610000.0, 110000000.0, closed='right'),
 Interval(1086000.0, 1610000.0, closed='right'),
 Interval(791000.0, 1086000.0, closed='right'),
 Interval(574000.0, 791000.0, closed='right'),
 Interval(25999.999, 574000.0, closed='right')]

In [ ]:
def quant_eval(df, q_val, modl_typ, feats=crit_cols):

  qdf = df[df["PriceRank"] == q_val]
  qdf.reset_index(drop=True, inplace=True)
  ct = len(qdf)

  wk4, wk5 = W4_Regression(qdf), W5_Regs(qdf)
  wk7, wk8 = W7_Boosting(qdf), W8_Eval(qdf)
  train, test = wk4.test_train_split()

  qlog_df = wk7.log_transform()
  w4, w5 = W4_Regression(qlog_df), W5_Regs(qlog_df)
  w7, w8 = W7_Boosting(qlog_df), W8_Eval(qlog_df)
  tr, te = w4.test_train_split()


  if modl_typ == "Linear Regression":              # Linear Regression
    r2 = wk5.shrt_main(wk4.LinReg, train, test, wk4.r2_eval, crit_cols, prt=False)
    mape = wk5.shrt_main(wk4.LinReg, train, test, wk8.mape_eval, crit_cols, rev=False, prt=False)
    mdape = wk5.shrt_main(wk4.LinReg, train, test, wk8.mdape_eval, crit_cols, rev=False, prt=False)

    lg_r2 = w5.shrt_main(w4.LinReg, tr, te, w4.r2_eval, crit_cols, prt=False)
    lg_mape = w5.shrt_main(w4.LinReg, tr, te, w8.mape_eval, crit_cols, rev=False, prt=False)
    lg_mdape = w5.shrt_main(w4.LinReg, tr, te, w8.mdape_eval, crit_cols, rev=False, prt=False)


  if modl_typ == "Decision Tree Regressor":             # Decision Tree Regressor
    r2 = wk5.shrt_main(wk5.TreeReg, train, test, wk4.r2_eval, crit_cols, prt=False)
    mape = wk5.shrt_main(wk5.TreeReg, train, test, wk8.mape_eval, crit_cols, rev=False, prt=False)
    mdape = wk5.shrt_main(wk5.TreeReg, train, test, wk8.mdape_eval, crit_cols, rev=False, prt=False)

    lg_r2 = w5.shrt_main(w5.TreeReg, tr, te, w4.r2_eval, crit_cols, prt=False)
    lg_mape = w5.shrt_main(w5.TreeReg, tr, te, w8.mape_eval, crit_cols, rev=False, prt=False)
    lg_mdape = w5.shrt_main(w5.TreeReg, tr, te, w8.mdape_eval, crit_cols, rev=False, prt=False)


  if modl_typ == "Random Forest Regressor":             # Random Forest Regressor
    r2 = wk5.shrt_main(wk5.ForestReg, train, test, wk4.r2_eval, crit_cols, prt=False)
    mape = wk5.shrt_main(wk5.ForestReg, train, test, wk8.mape_eval, crit_cols, rev=False, prt=False)
    mdape = wk5.shrt_main(wk5.ForestReg, train, test, wk8.mdape_eval, crit_cols, rev=False, prt=False)

    lg_r2 = w5.shrt_main(w5.ForestReg, tr, te, w4.r2_eval, crit_cols, prt=False)
    lg_mape = w5.shrt_main(w5.ForestReg, tr, te, w8.mape_eval, crit_cols, rev=False, prt=False)
    lg_mdape = w5.shrt_main(w5.ForestReg, tr, te, w8.mdape_eval, crit_cols, rev=False, prt=False)


  if modl_typ == "XGBoost":             # XGBoost

    r2 = wk5.shrt_main(wk7.XGB, train, test, wk4.r2_eval, crit_cols, prt=False,)
    mape = wk5.shrt_main(wk7.XGB, train, test, wk8.mape_eval, crit_cols, rev=False, prt=False,
                         dep=bst_r2s["max_depth"].iloc[0], l_rate=bst_r2s["learning_rate"].iloc[0], est=bst_r2s["n_estimators"].iloc[0])
    mdape = wk5.shrt_main(wk7.XGB, train, test, wk8.mdape_eval, crit_cols, rev=False, prt=False,
                          dep=bst_r2s["max_depth"].iloc[0], l_rate=bst_r2s["learning_rate"].iloc[0], est=bst_r2s["n_estimators"].iloc[0])

    lg_r2 = w5.shrt_main(w7.XGB, tr, te, w4.r2_eval, crit_cols, prt=False)
    lg_mape = w5.shrt_main(w7.XGB, tr, te, w8.mape_eval, crit_cols, rev=False, prt=False,
                           dep=bst_r2s["max_depth"].iloc[1], l_rate=bst_r2s["learning_rate"].iloc[1], est=bst_r2s["n_estimators"].iloc[1])
    lg_mdape = w5.shrt_main(w7.XGB, tr, te, w8.mdape_eval, crit_cols, rev=False, prt=False,
                            dep=bst_r2s["max_depth"].iloc[1], l_rate=bst_r2s["learning_rate"].iloc[1], est=bst_r2s["n_estimators"].iloc[1])


  if modl_typ == "Gradient Boosting Regressor":             # Gradient Boosting Regressor

    r2 = wk5.shrt_main(wk7.GBR, train, test, wk4.r2_eval, crit_cols, prt=False)
    mape = wk5.shrt_main(wk7.GBR, train, test, wk8.mape_eval, crit_cols, rev=False, prt=False,
                         dep=bst_r2s["max_depth"].iloc[2], l_rate=bst_r2s["learning_rate"].iloc[2], est=bst_r2s["n_estimators"].iloc[2])
    mdape = wk5.shrt_main(wk7.GBR, train, test, wk8.mdape_eval, crit_cols, rev=False, prt=False,
                          dep=bst_r2s["max_depth"].iloc[2], l_rate=bst_r2s["learning_rate"].iloc[2], est=bst_r2s["n_estimators"].iloc[2])

    lg_r2 = w5.shrt_main(w7.GBR, tr, te, w4.r2_eval, crit_cols, prt=False)
    lg_mape = w5.shrt_main(w7.GBR, tr, te, w8.mape_eval, crit_cols, rev=False, prt=False,
                           dep=bst_r2s["max_depth"].iloc[3], l_rate=bst_r2s["learning_rate"].iloc[3], est=bst_r2s["n_estimators"].iloc[3])
    lg_mdape = w5.shrt_main(w7.GBR, tr, te, w8.mdape_eval, crit_cols, rev=False, prt=False,
                            dep=bst_r2s["max_depth"].iloc[3], l_rate=bst_r2s["learning_rate"].iloc[3], est=bst_r2s["n_estimators"].iloc[3])


  if modl_typ == "LightGBM":             # LightGBM

    r2 = wk5.shrt_main(wk7.L_GBM, train, test, wk4.r2_eval, crit_cols, prt=False)
    mape = wk5.shrt_main(wk7.L_GBM, train, test, wk8.mape_eval, crit_cols, rev=False, prt=False,
                         dep=bst_r2s["max_depth"].iloc[4], l_rate=bst_r2s["learning_rate"].iloc[4], est=bst_r2s["n_estimators"].iloc[4])
    mdape = wk5.shrt_main(wk7.L_GBM, train, test, wk8.mdape_eval, crit_cols, rev=False, prt=False,
                          dep=bst_r2s["max_depth"].iloc[4], l_rate=bst_r2s["learning_rate"].iloc[4], est=bst_r2s["n_estimators"].iloc[4])

    lg_r2 = w5.shrt_main(w7.L_GBM, tr, te, w4.r2_eval, crit_cols, prt=False)
    lg_mape = w5.shrt_main(w7.L_GBM, tr, te, w8.mape_eval, crit_cols, rev=False, prt=False,
                           dep=bst_r2s["max_depth"].iloc[5], l_rate=bst_r2s["learning_rate"].iloc[5], est=bst_r2s["n_estimators"].iloc[5])
    lg_mdape = w5.shrt_main(w7.L_GBM, tr, te, w8.mdape_eval, crit_cols, rev=False, prt=False,
                            dep=bst_r2s["max_depth"].iloc[5], l_rate=bst_r2s["learning_rate"].iloc[5], est=bst_r2s["n_estimators"].iloc[5])


  if modl_typ == "Histogram-based Gradient Boosting Regressor":             # Histogram-based Gradient Boosting Regressor

    r2 = wk5.shrt_main(wk7.HGBR, train, test, wk4.r2_eval, crit_cols, prt=False)
    mape = wk5.shrt_main(wk7.HGBR, train, test, wk8.mape_eval, crit_cols, rev=False, prt=False,
                         dep=bst_r2s["max_depth"].iloc[6], l_rate=bst_r2s["learning_rate"].iloc[6])
    mdape = wk5.shrt_main(wk7.HGBR, train, test, wk8.mdape_eval, crit_cols, rev=False, prt=False,
                          dep=bst_r2s["max_depth"].iloc[6], l_rate=bst_r2s["learning_rate"].iloc[6])

    lg_r2 = w5.shrt_main(w7.HGBR, tr, te, w4.r2_eval, crit_cols, prt=False)
    lg_mape = w5.shrt_main(w7.HGBR, tr, te, w8.mape_eval, crit_cols, rev=False, prt=False,
                           dep=bst_r2s["max_depth"].iloc[7], l_rate=bst_r2s["learning_rate"].iloc[7])
    lg_mdape = w5.shrt_main(w7.HGBR, tr, te, w8.mdape_eval, crit_cols, rev=False, prt=False,
                            dep=bst_r2s["max_depth"].iloc[7], l_rate=bst_r2s["learning_rate"].iloc[7])

  new_df = pd.DataFrame({"Model": [modl_typ]*2, "PriceRank": q_val, "RowCount": ct, "LogForm": [False,True], "R2 Score": [r2[1],lg_r2[1]], "MAPE": [mape[1],lg_mape[1]], "MdAPE": [mdape[1],lg_mdape[1]]})
  return new_df

In [ ]:
qdfLR1 = quant_eval(enr_df, "Low", "Linear Regression")

In [ ]:
qdfLR2 = quant_eval(enr_df, "Low-Medium", "Linear Regression")

In [ ]:
qdfLR3 = quant_eval(enr_df, "Medium", "Linear Regression")

In [ ]:
qdfLR4 = quant_eval(enr_df, "Medium-High", "Linear Regression")

In [ ]:
qdfLR5 = quant_eval(enr_df, "High", "Linear Regression")

In [ ]:
qdfLR = pd.concat([qdfLR1, qdfLR2, qdfLR3, qdfLR4, qdfLR5])
qdfLR.sort_values(by=["LogForm"], ignore_index=True, inplace=True)
qdfLR

,Model,PriceRank,RowCount,LogForm,R2 Score,MAPE,MdAPE
0,Linear Regression,Low,20417,False,0.718498,1.321526e-01,0.079989
1,Linear Regression,Low-Medium,20416,False,0.190363,7.424308e-02,0.070717
2,Linear Regression,Medium,20412,False,0.182600,6.319476e-02,0.057480
3,Linear Regression,Medium-High,20499,False,0.232214,8.058203e-02,0.071625
4,Linear Regression,High,20327,False,0.679001,2.337879e-01,0.189646
5,Linear Regression,Low,20417,True,1.000000,7.241376e-17,0.000000
6,Linear Regression,Low-Medium,20416,True,1.000000,5.976146e-17,0.000000
7,Linear Regression,Medium,20412,True,1.000000,3.428558e-17,0.000000
8,Linear Regression,Medium-High,20499,True,1.000000,5.952695e-17,0.000000
9,Linear Regression,High,20327,True,1.000000,4.842073e-17,0.000000


In [ ]:
qdfDTR1 = quant_eval(enr_df, "Low", "Decision Tree Regressor")

In [ ]:
qdfDTR2 = quant_eval(enr_df, "Low-Medium", "Decision Tree Regressor")

In [ ]:
qdfDTR3 = quant_eval(enr_df, "Medium", "Decision Tree Regressor")

In [ ]:
qdfDTR4 = quant_eval(enr_df, "Medium-High", "Decision Tree Regressor")

In [ ]:
qdfDTR5 = quant_eval(enr_df, "High", "Decision Tree Regressor")

In [ ]:
qdfDTR = pd.concat([qdfDTR1, qdfDTR2, qdfDTR3, qdfDTR4, qdfDTR5])
qdfDTR.sort_values(by=["LogForm"], ignore_index=True, inplace=True)
qdfDTR

,Model,PriceRank,RowCount,LogForm,R2 Score,MAPE,MdAPE
0,Decision Tree Regressor,Low,20417,False,0.999212,0.001204,0.000000e+00
1,Decision Tree Regressor,Low-Medium,20416,False,0.999981,0.000073,0.000000e+00
2,Decision Tree Regressor,Medium,20412,False,0.999987,0.000060,0.000000e+00
3,Decision Tree Regressor,Medium-High,20499,False,0.999829,0.000173,0.000000e+00
4,Decision Tree Regressor,High,20327,False,0.985346,0.002049,0.000000e+00
5,Decision Tree Regressor,Low,20417,True,0.998886,0.000107,1.393079e-16
6,Decision Tree Regressor,Low-Medium,20416,True,0.999972,0.000006,1.335136e-16
7,Decision Tree Regressor,Medium,20412,True,0.999987,0.000004,1.933676e-16
8,Decision Tree Regressor,Medium-High,20499,True,0.999790,0.000013,2.501177e-16
9,Decision Tree Regressor,High,20327,True,0.998761,0.000140,2.432704e-16


In [ ]:
qdfRFR1 = quant_eval(enr_df, "Low", "Random Forest Regressor")

In [ ]:
qdfRFR2 = quant_eval(enr_df, "Low-Medium", "Random Forest Regressor")

In [ ]:
qdfRFR3 = quant_eval(enr_df, "Medium", "Random Forest Regressor")

In [ ]:
qdfRFR4 = quant_eval(enr_df, "Medium-High", "Random Forest Regressor")

In [ ]:
qdfRFR5 = quant_eval(enr_df, "High", "Random Forest Regressor")

In [ ]:
qdfRFR = pd.concat([qdfRFR1, qdfRFR2, qdfRFR3, qdfRFR4, qdfRFR5])
qdfRFR.sort_values(by=["LogForm"], ignore_index=True, inplace=True)
qdfRFR

,Model,PriceRank,RowCount,LogForm,R2 Score,MAPE,MdAPE
0,Random Forest Regressor,Low,20417,False,0.999095,0.002032,0.000000e+00
1,Random Forest Regressor,Low-Medium,20416,False,0.999985,0.000070,0.000000e+00
2,Random Forest Regressor,Medium,20412,False,0.999913,0.000112,0.000000e+00
3,Random Forest Regressor,Medium-High,20499,False,0.999763,0.000174,0.000000e+00
4,Random Forest Regressor,High,20327,False,0.984631,0.001836,7.037983e-05
5,Random Forest Regressor,Low,20417,True,0.999159,0.000114,1.620064e-09
6,Random Forest Regressor,Low-Medium,20416,True,0.999983,0.000005,2.253588e-15
7,Random Forest Regressor,Medium,20412,True,0.999684,0.000012,1.958527e-15
8,Random Forest Regressor,Medium-High,20499,True,0.999741,0.000012,2.023384e-15
9,Random Forest Regressor,High,20327,True,0.999014,0.000116,4.542826e-06


In [ ]:
qdfXGB1 = quant_eval(enr_df, "Low", "XGBoost")

In [ ]:
qdfXGB2 = quant_eval(enr_df, "Low-Medium", "XGBoost")

In [ ]:
qdfXGB3 = quant_eval(enr_df, "Medium", "XGBoost")

In [ ]:
qdfXGB4 = quant_eval(enr_df, "Medium-High", "XGBoost")

In [ ]:
qdfXGB5 = quant_eval(enr_df, "High", "XGBoost")

In [ ]:
qdfXGB = pd.concat([qdfXGB1, qdfXGB2, qdfXGB3, qdfXGB4, qdfXGB5])
qdfXGB.sort_values(by=["LogForm"], ignore_index=True, inplace=True)
qdfXGB

,Model,PriceRank,RowCount,LogForm,R2 Score,MAPE,MdAPE
0,XGBoost,Low,20417,False,0.998935,0.004008,0.001320
1,XGBoost,Low-Medium,20416,False,0.998515,0.000918,0.000061
2,XGBoost,Medium,20412,False,0.996666,0.001068,0.000278
3,XGBoost,Medium-High,20499,False,0.998370,0.001351,0.000444
4,XGBoost,High,20327,False,0.964851,0.005437,0.002638
5,XGBoost,Low,20417,True,0.995840,0.000490,0.000185
6,XGBoost,Low-Medium,20416,True,0.998574,0.000083,0.000015
7,XGBoost,Medium,20412,True,0.996470,0.000082,0.000021
8,XGBoost,Medium-High,20499,True,0.998547,0.000101,0.000036
9,XGBoost,High,20327,True,0.998700,0.000324,0.000138


In [ ]:
qdfGBR1 = quant_eval(enr_df, "Low", "Gradient Boosting Regressor")

In [ ]:
qdfGBR2 = quant_eval(enr_df, "Low-Medium", "Gradient Boosting Regressor")

In [ ]:
qdfGBR3 = quant_eval(enr_df, "Medium", "Gradient Boosting Regressor")

In [ ]:
qdfGBR4 = quant_eval(enr_df, "Medium-High", "Gradient Boosting Regressor")

In [ ]:
qdfGBR5 = quant_eval(enr_df, "High", "Gradient Boosting Regressor")

In [ ]:
qdfGBR = pd.concat([qdfGBR1, qdfGBR2, qdfGBR3, qdfGBR4, qdfGBR5])
qdfGBR.sort_values(by=["LogForm"], ignore_index=True, inplace=True)
qdfGBR

,Model,PriceRank,RowCount,LogForm,R2 Score,MAPE,MdAPE
0,Gradient Boosting Regressor,Low,20417,False,0.991478,0.012769,0.006652
1,Gradient Boosting Regressor,Low-Medium,20416,False,0.997418,0.002946,0.001720
2,Gradient Boosting Regressor,Medium,20412,False,0.995346,0.002914,0.001821
3,Gradient Boosting Regressor,Medium-High,20499,False,0.994916,0.003861,0.002927
4,Gradient Boosting Regressor,High,20327,False,0.981653,0.020284,0.015454
5,Gradient Boosting Regressor,Low,20417,True,0.993093,0.000339,0.000184
6,Gradient Boosting Regressor,Low-Medium,20416,True,0.997147,0.000021,0.000005
7,Gradient Boosting Regressor,Medium,20412,True,0.996443,0.000037,0.000008
8,Gradient Boosting Regressor,Medium-High,20499,True,0.995503,0.000043,0.000018
9,Gradient Boosting Regressor,High,20327,True,0.997678,0.000278,0.000125


In [ ]:
qdfLGBM1 = quant_eval(enr_df, "Low", "LightGBM")

In [ ]:
qdfLGBM2 = quant_eval(enr_df, "Low-Medium", "LightGBM")

In [ ]:
qdfLGBM3 = quant_eval(enr_df, "Medium", "LightGBM")

In [ ]:
qdfLGBM4 = quant_eval(enr_df, "Medium-High", "LightGBM")

In [ ]:
qdfLGBM5 = quant_eval(enr_df, "High", "LightGBM")

In [ ]:
qdfLGBM = pd.concat([qdfLGBM1, qdfLGBM2, qdfLGBM3, qdfLGBM4, qdfLGBM5])
qdfLGBM.sort_values(by=["LogForm"], ignore_index=True, inplace=True)
qdfLGBM

,Model,PriceRank,RowCount,LogForm,R2 Score,MAPE,MdAPE
0,LightGBM,Low,20417,False,0.998300,0.006893,0.002703
1,LightGBM,Low-Medium,20416,False,0.998716,0.001391,0.000121
2,LightGBM,Medium,20412,False,0.999257,0.001585,0.000315
3,LightGBM,Medium-High,20499,False,0.998061,0.002075,0.000845
4,LightGBM,High,20327,False,0.975190,0.018192,0.013502
5,LightGBM,Low,20417,True,0.998173,0.000645,0.000311
6,LightGBM,Low-Medium,20416,True,0.998821,0.000111,0.000009
7,LightGBM,Medium,20412,True,0.999292,0.000115,0.000024
8,LightGBM,Medium-High,20499,True,0.998172,0.000150,0.000054
9,LightGBM,High,20327,True,0.998370,0.000525,0.000279


In [ ]:
qdfHGBR1 = quant_eval(enr_df, "Low", "Histogram-based Gradient Boosting Regressor")

In [ ]:
qdfHGBR2 = quant_eval(enr_df, "Low-Medium", "Histogram-based Gradient Boosting Regressor")

In [ ]:
qdfHGBR3 = quant_eval(enr_df, "Medium", "Histogram-based Gradient Boosting Regressor")

In [ ]:
qdfHGBR4 = quant_eval(enr_df, "Medium-High", "Histogram-based Gradient Boosting Regressor")

In [ ]:
qdfHGBR5 = quant_eval(enr_df, "High", "Histogram-based Gradient Boosting Regressor")

In [ ]:
qdfHGBR = pd.concat([qdfHGBR1, qdfHGBR2, qdfHGBR3, qdfHGBR4, qdfHGBR5])
qdfHGBR.sort_values(by=["LogForm"], ignore_index=True, inplace=True)
qdfHGBR

,Model,PriceRank,RowCount,LogForm,R2 Score,MAPE,MdAPE
0,Histogram-based Gradient Boosting Regressor,Low,20417,False,0.998299,0.012097,0.004809
1,Histogram-based Gradient Boosting Regressor,Low-Medium,20416,False,0.997975,0.001810,0.000821
2,Histogram-based Gradient Boosting Regressor,Medium,20412,False,0.997478,0.002007,0.000660
3,Histogram-based Gradient Boosting Regressor,Medium-High,20499,False,0.998036,0.002356,0.001328
4,Histogram-based Gradient Boosting Regressor,High,20327,False,0.964200,0.033174,0.023478
5,Histogram-based Gradient Boosting Regressor,Low,20417,True,0.993146,0.000723,0.000350
6,Histogram-based Gradient Boosting Regressor,Low-Medium,20416,True,0.997766,0.000125,0.000066
7,Histogram-based Gradient Boosting Regressor,Medium,20412,True,0.997366,0.000139,0.000048
8,Histogram-based Gradient Boosting Regressor,Medium-High,20499,True,0.998250,0.000151,0.000074
9,Histogram-based Gradient Boosting Regressor,High,20327,True,0.998371,0.000615,0.000355


In [ ]:
qdf_all = pd.concat([qdfLR, qdfDTR, qdfRFR, qdfXGB, qdfGBR, qdfLGBM, qdfHGBR])
qdf_all

,Model,PriceRank,RowCount,LogForm,R2 Score,MAPE,MdAPE
0,Linear Regression,Low,20417,False,0.718498,0.132153,0.079989
1,Linear Regression,Low-Medium,20416,False,0.190363,0.074243,0.070717
2,Linear Regression,Medium,20412,False,0.182600,0.063195,0.057480
3,Linear Regression,Medium-High,20499,False,0.232214,0.080582,0.071625
4,Linear Regression,High,20327,False,0.679001,0.233788,0.189646
...,...,...,...,...,...,...,...
5,Histogram-based Gradient Boosting Regressor,Low,20417,True,0.993146,0.000723,0.000350
6,Histogram-based Gradient Boosting Regressor,Low-Medium,20416,True,0.997766,0.000125,0.000066
7,Histogram-based Gradient Boosting Regressor,Medium,20412,True,0.997366,0.000139,0.000048
8,Histogram-based Gradient Boosting Regressor,Medium-High,20499,True,0.998250,0.000151,0.000074


In [ ]:
save_csv(qdf_all, "price_band_metrics.csv")

# **Metrics DataFrame: *R2, MAPE, MdAPE***

 * R2 Score (R2)
 * Mean Absolute Percentage Error (MAPE)
 * Median Absolute Percentage Error (MdAPE)

In [ ]:
reg_r2s = load_data("regression_r2s.csv")

In [ ]:
modl = ["Linear Regression"]*4 + ["Decision Tree Regressor"]*4 + ["Random Forest Regressor"]*4 + [
    "XGBoost"]*4 + ["Gradient Boosting Regressor"]*4 + ["LightGBM"]*4 + ["Histogram-based Gradient Boosting Regressor"]*4

lg = [False, True]*14

typ = ["MAPE", "MAPE", "MdAPE", "MdAPE"]*7

cols = [c_LRmape, c_logLRmape, c_LRmdape, c_logLRmdape] + [c_DTRmape, c_logDTRmape, c_DTRmdape, c_logDTRmdape] + [
    c_RFRmape, c_logRFRmape, c_RFRmdape, c_logRFRmdape] + [c_XGBmape, c_logXGBmape, c_XGBmdape, c_logXGBmdape] + [
        c_GBRmape, c_logGBRmape, c_GBRmdape, c_logGBRmdape] + [c_LGBMmape, c_logLGBMmape, c_LGBMmdape, c_logLGBMmdape] + [
        c_HGBRmape, c_logHGBRmape, c_HGBRmdape, c_logHGBRmdape]

scrs = [LRmape, logLRmape, LRmdape, logLRmdape] + [DTRmape, logDTRmape, DTRmdape, logDTRmdape] + [RFRmape, logRFRmape, RFRmdape, logRFRmdape] + [
    XGBmape, logXGBmape, XGBmdape, logXGBmdape] + [GBRmape, logGBRmape, GBRmdape, logGBRmdape] + [LGBMmape, logLGBMmape, LGBMmdape, logLGBMmdape] + [
        HGBRmape, logHGBRmape, HGBRmdape, logHGBRmdape]

In [ ]:
md = [None]*12 + [bst_r2s["max_depth"].iloc[i] for i in range(2)]*2 + [bst_r2s["max_depth"].iloc[i] for i in range(2,4)]*2 + [
    bst_r2s["max_depth"].iloc[i] for i in range(4,6)]*2 + [bst_r2s["max_depth"].iloc[i] for i in range(6,8)]*2

lr = [None]*12 + [bst_r2s["learning_rate"].iloc[i] for i in range(2)]*2 + [bst_r2s["learning_rate"].iloc[i] for i in range(2,4)]*2 + [
    bst_r2s["learning_rate"].iloc[i] for i in range(4,6)]*2 + [bst_r2s["learning_rate"].iloc[i] for i in range(6,8)]*2

ne = [None]*12 + [bst_r2s["n_estimators"].iloc[i] for i in range(2)]*2 + [bst_r2s["n_estimators"].iloc[i] for i in range(2,4)]*2 + [
    bst_r2s["n_estimators"].iloc[i] for i in range(4,6)]*2 + [bst_r2s["n_estimators"].iloc[i] for i in range(6,8)]*2

In [ ]:
mets = {"Model": modl, "LogForm": lg, "ScoreType": typ, "ScoreValue": scrs, "max_depth": md, "learning_rate": lr, "n_estimators": ne, "columns": cols}

met_df = pd.DataFrame(mets)
met_df = pd.concat([reg_r2s, bst_r2s, met_df])
met_df.sort_values(by=["Model", "ScoreType", "LogForm"], ignore_index=True, inplace=True)
met_df

In [ ]:
save_csv(met_df, "metrics_summary.csv")